# Notebook 02 — Scaling study

End-to-end scaling run across the full grid `N × p × seed = 5 × 3 × 5 = 75 experiments`.

> **Runtime note:** on an M4 MacBook Air this takes ≈3–4 min. If the study is already on disk at `results/data/scaling_results.json`, re-running is optional — the aggregation cell works off the saved file.

## Cell 1 — Load config

In [ ]:
import os, sys
from pathlib import Path
# Move CWD to the project root so relative paths (configs/, results/) work
# regardless of where the notebook is launched from.
_here = Path.cwd()
if _here.name == 'notebooks':
    os.chdir(_here.parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print('cwd:', Path.cwd())

from src.run_experiment import load_config

cfg = load_config()
print(f'graph_sizes = {cfg["graph_sizes"]}')
print(f'p_values    = {cfg["p_values"]}')
print(f'seeds       = {cfg["seeds"]}')

## Cell 2 — Run full experiment grid

Comment/uncomment the `run_scaling_study(...)` call depending on whether you want a fresh run or to load the saved one.

In [ ]:
from pathlib import Path
import pandas as pd
from src.run_experiment import run_scaling_study

saved = Path('results/data/scaling_results.json')
if saved.exists():
    print(f'loading {saved} ...')
    df = pd.read_json(saved, orient='records')
else:
    print('running full scaling study (≈3 min) ...')
    df = run_scaling_study(cfg, verbose=True)

print(f'loaded {len(df)} experiments')
df.head()

## Cell 3 — Aggregate results

In [ ]:
from src.run_experiment import aggregate_results

agg = aggregate_results(df)
agg.round(4)

## Cell 4 — Save CSVs

In [ ]:
from src.run_experiment import save_results

paths = save_results(df, output_dir='results/data', stem='scaling_results',
                     formats=('csv', 'json'))
for fmt, p in paths.items():
    print(f'{fmt:>4} -> {p}')

agg_path = 'results/data/aggregated_results.csv'
agg.to_csv(agg_path, index=False)
print(f' csv -> {agg_path}')

## Cell 5 — Summary table

In [ ]:
print('Per (N, p) summary (mean ± std across 5 seeds):')
summary = agg[['n', 'p', 'r_mean', 'r_std', 'cnot_mean', 'depth_mean', 'n_seeds']]
print(summary.to_string(index=False))

# Farhi bound check — every 3-regular p=1 row must satisfy r ≥ 0.6924.
p1_3reg = df[(df['p'] == 1) & (df['n'] >= 4)]
assert (p1_3reg['approximation_ratio'] >= 0.6924).all()
print(f'\n✓ Farhi 2014 bound (r ≥ 0.6924) holds on all {len(p1_3reg)} 3-regular p=1 rows')